# 🛒 Retail Sales Business Analysis using SQL
**Author:** Khin Me Me Zaw  
**Tools:** Python, SQLite, Pandas, Matplotlib  
**Dataset:** Online Retail Dataset (UCI / Kaggle)  
**Goal:** Answer real business questions using SQL queries on retail transaction data.

---
## 📌 Business Questions We Will Answer
1. What is the total revenue by country?
2. Which are the top 10 best-selling products?
3. What is the monthly revenue trend?
4. Which customers generate the most revenue? (Top 10)
5. What is the average order value per country?
6. Which products have the highest return/cancellation rate?
7. What are the peak sales hours of the day?
8. Which month has the lowest sales? (Seasonal insight)
9. How many unique customers per country?
10. RFM Segmentation — who are the most valuable customers?

## ⚙️ Step 1: Setup — Install Libraries & Download Dataset

In [ ]:
# Install required libraries
!pip install pandas matplotlib seaborn -q

# Download the Online Retail dataset from UCI
!wget -q 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx' -O online_retail.xlsx
print('Dataset downloaded!')

## 📥 Step 2: Load Data into SQLite Database

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load Excel into DataFrame
df = pd.read_excel('online_retail.xlsx', dtype={'CustomerID': str})
print(f'Raw data shape: {df.shape}')
df.head()

In [ ]:
# Basic data cleaning
df.dropna(subset=['CustomerID'], inplace=True)     # Remove rows with no customer
df = df[df['Quantity'] > 0]                        # Remove returns/cancellations for main analysis
df = df[df['UnitPrice'] > 0]                       # Remove zero-price items
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']   # Create Revenue column
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['Hour'] = df['InvoiceDate'].dt.hour

print(f'Clean data shape: {df.shape}')
print(f'Date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')
df.info()

In [ ]:
# Load into SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('retail', conn, index=False, if_exists='replace')
print('Data loaded into SQLite successfully!')

# Helper function to run SQL and return DataFrame
def sql(query):
    return pd.read_sql_query(query, conn)

---
## 📊 Q1: Total Revenue by Country (Top 10)

In [ ]:
q1 = sql("""
    SELECT
        Country,
        ROUND(SUM(Revenue), 2) AS Total_Revenue,
        COUNT(DISTINCT CustomerID) AS Unique_Customers,
        COUNT(DISTINCT InvoiceNo) AS Total_Orders
    FROM retail
    GROUP BY Country
    ORDER BY Total_Revenue DESC
    LIMIT 10
""")
print(q1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(q1['Country'][::-1], q1['Total_Revenue'][::-1], color='#16613E')
ax.set_xlabel('Total Revenue (GBP)')
ax.set_title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('q1_revenue_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: The UK dominates revenue. Consider if international expansion is viable.')

---
## 📦 Q2: Top 10 Best-Selling Products (by Revenue)

In [ ]:
q2 = sql("""
    SELECT
        Description AS Product,
        SUM(Quantity) AS Total_Units_Sold,
        ROUND(SUM(Revenue), 2) AS Total_Revenue
    FROM retail
    WHERE Description IS NOT NULL
    GROUP BY Description
    ORDER BY Total_Revenue DESC
    LIMIT 10
""")
print(q2.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(range(len(q2)), q2['Total_Revenue'], color='#1A6640')
ax.set_xticks(range(len(q2)))
ax.set_xticklabels([p[:30] for p in q2['Product']], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Total Revenue (GBP)')
ax.set_title('Top 10 Best-Selling Products by Revenue', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('q2_top_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: A small number of products drive the majority of revenue — classic 80/20 rule.')

---
## 📅 Q3: Monthly Revenue Trend

In [ ]:
q3 = sql("""
    SELECT
        strftime('%Y-%m', InvoiceDate) AS Month,
        ROUND(SUM(Revenue), 2) AS Monthly_Revenue,
        COUNT(DISTINCT InvoiceNo) AS Orders
    FROM retail
    GROUP BY Month
    ORDER BY Month
""")
print(q3.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.fill_between(q3['Month'], q3['Monthly_Revenue'], alpha=0.3, color='#16613E')
ax1.plot(q3['Month'], q3['Monthly_Revenue'], color='#16613E', linewidth=2, marker='o')
ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue (GBP)', color='#16613E')
ax1.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('q3_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: Identify seasonal peaks (e.g. Nov-Dec holiday season) and plan inventory accordingly.')

---
## 👑 Q4: Top 10 Customers by Revenue

In [ ]:
q4 = sql("""
    SELECT
        CustomerID,
        Country,
        ROUND(SUM(Revenue), 2) AS Total_Spent,
        COUNT(DISTINCT InvoiceNo) AS Total_Orders,
        ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS Avg_Order_Value
    FROM retail
    GROUP BY CustomerID
    ORDER BY Total_Spent DESC
    LIMIT 10
""")
print(q4.to_string(index=False))
print('\n💡 Insight: Top 10 customers may represent a significant % of revenue — consider VIP loyalty strategies.')

---
## 💰 Q5: Average Order Value by Country

In [ ]:
q5 = sql("""
    SELECT
        Country,
        ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS Avg_Order_Value,
        COUNT(DISTINCT InvoiceNo) AS Total_Orders
    FROM retail
    GROUP BY Country
    HAVING Total_Orders >= 10
    ORDER BY Avg_Order_Value DESC
    LIMIT 10
""")
print(q5.to_string(index=False))
print('\n💡 Insight: Some countries order less frequently but spend more per order — a different marketing approach is needed.')

---
## ❌ Q6: Products with Highest Cancellation Rate

In [ ]:
# For this query we reload original (with cancellations)
df_raw = pd.read_excel('online_retail.xlsx', dtype={'CustomerID': str})
df_raw.dropna(subset=['CustomerID', 'Description'], inplace=True)
df_raw.to_sql('retail_raw', conn, index=False, if_exists='replace')

q6 = sql("""
    SELECT
        Description AS Product,
        SUM(CASE WHEN Quantity < 0 THEN 1 ELSE 0 END) AS Cancellations,
        SUM(CASE WHEN Quantity > 0 THEN 1 ELSE 0 END) AS Sales,
        ROUND(
            100.0 * SUM(CASE WHEN Quantity < 0 THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN Quantity > 0 THEN 1 ELSE 0 END), 0)
        , 1) AS Cancellation_Rate_Pct
    FROM retail_raw
    GROUP BY Description
    HAVING Sales >= 20
    ORDER BY Cancellation_Rate_Pct DESC
    LIMIT 10
""")
print(q6.to_string(index=False))
print('\n💡 Insight: High cancellation rate may signal quality issues, misleading descriptions, or delivery problems.')

---
## ⏰ Q7: Peak Sales Hours

In [ ]:
q7 = sql("""
    SELECT
        CAST(strftime('%H', InvoiceDate) AS INTEGER) AS Hour,
        COUNT(DISTINCT InvoiceNo) AS Orders,
        ROUND(SUM(Revenue), 2) AS Revenue
    FROM retail
    GROUP BY Hour
    ORDER BY Hour
""")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(q7['Hour'], q7['Orders'], color='#16613E', alpha=0.85)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Orders')
ax.set_title('Peak Sales Hours of the Day', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('q7_peak_hours.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: Most orders happen mid-morning (10am-12pm). Schedule email campaigns and promotions around peak hours.')

---
## 📉 Q8: Lowest Sales Month

In [ ]:
q8 = sql("""
    SELECT
        strftime('%Y-%m', InvoiceDate) AS Month,
        ROUND(SUM(Revenue), 2) AS Monthly_Revenue,
        COUNT(DISTINCT InvoiceNo) AS Orders
    FROM retail
    GROUP BY Month
    ORDER BY Monthly_Revenue ASC
    LIMIT 5
""")
print('Lowest Revenue Months:')
print(q8.to_string(index=False))
print('\n💡 Insight: These are the slowest months — good time to plan promotions, discounts, or new product launches.')

---
## 🌍 Q9: Unique Customers per Country

In [ ]:
q9 = sql("""
    SELECT
        Country,
        COUNT(DISTINCT CustomerID) AS Unique_Customers
    FROM retail
    GROUP BY Country
    ORDER BY Unique_Customers DESC
    LIMIT 10
""")
print(q9.to_string(index=False))
print('\n💡 Insight: Customer base is heavily UK-centric. Growing international customers could diversify revenue.')

---
## 🏅 Q10: RFM Customer Segmentation
**RFM = Recency · Frequency · Monetary**  
A classic method to identify your most valuable customers.

In [ ]:
rfm = sql("""
    SELECT
        CustomerID,
        -- Recency: days since last purchase (lower = better)
        CAST(julianday('2011-12-10') - julianday(MAX(InvoiceDate)) AS INTEGER) AS Recency,
        -- Frequency: number of unique orders
        COUNT(DISTINCT InvoiceNo) AS Frequency,
        -- Monetary: total revenue
        ROUND(SUM(Revenue), 2) AS Monetary
    FROM retail
    GROUP BY CustomerID
""")

# Score each dimension 1-4 using quartiles
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal Customers'
    elif score >= 6: return 'Potential Loyalists'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(segment)
segment_summary = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1).reset_index().sort_values('Avg_Monetary', ascending=False)

print(segment_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#16613E', '#2E8B57', '#66BB6A', '#FFB300', '#E53935']
ax.bar(segment_summary['Segment'], segment_summary['Customers'], color=colors)
ax.set_title('Customer Segments (RFM Analysis)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('q10_rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: Champions are your VIPs — reward them. At Risk customers need re-engagement campaigns.')

---
## ✅ Summary of Business Insights

| # | Question | Key Finding |
|---|---|---|
| 1 | Revenue by country | UK dominates; explore top 5 international markets |
| 2 | Best-selling products | Top 10 products drive majority of revenue (80/20 rule) |
| 3 | Monthly trend | Clear seasonal peak in Nov-Dec |
| 4 | Top customers | A few customers account for disproportionate revenue |
| 5 | Avg order value | Some markets order less but spend more |
| 6 | Cancellations | Specific products have high return rates — investigate quality |
| 7 | Peak hours | Orders peak 10am–12pm — best window for email campaigns |
| 8 | Slowest months | Plan promotions for slow months to smooth revenue |
| 9 | Customer base | Heavy UK concentration — diversification opportunity |
| 10 | RFM segments | Actionable customer tiers for targeted marketing |

---
*Project by Khin Me Me Zaw | github.com/KhinMeMeZaw | kmezaw1998@gmail.com*